# Việc 3b — soi ứng viên bằng FiftyOne (Colab)

Xem tận mắt các khung mà mạch trả về, kèm **hộp vật thể** của BTC và **chữ OCR**.

Mục đích: trả lời câu hỏi mà không con số nào trả lời được — *ứng viên hạng 1 sai ở chỗ nào?*

---
### Cần đính kèm ĐÚNG MỘT tệp

`<tên gói>.zip` sinh ra ở máy có dữ liệu bằng lệnh:
```powershell
python -u -m scripts.xuat_goi_soi --dev 25
python -u -m scripts.xuat_goi_soi --cau "bộ trống đỏ và cây đàn piano" --so 300
python -u -m scripts.xuat_goi_soi --video L23_V025
```
Tệp nằm ở `D:\aic-data\derived\goi_soi\<tên>.zip`, thường 20–50 MB.

**Không cần** mã nguồn dự án, không cần `frame_map.parquet`, không cần `.env`.
Gói đã chứa sẵn ảnh và mọi nhãn.


## 1. Cài đặt — chạy ô này, máy ảo sẽ TỰ KHỞI ĐỘNG LẠI

Colab dựng sẵn một bản Pillow; FiftyOne kéo về bản khác, và hai bản trộn vào nhau
làm `import fiftyone` chết ngay ở dòng đầu:

```
ImportError: cannot import name '_Ink' from 'PIL._typing'
```

Ô dưới cài FiftyOne, ép cài lại Pillow cho khớp, rồi **tự khởi động lại máy ảo**.
Thấy thông báo *"Your session crashed"* là **bình thường** — chạy tiếp ô số 2.


In [ ]:
!pip install -q fiftyone 2>&1 | tail -2
!pip install -q --force-reinstall --no-cache-dir pillow 2>&1 | tail -2

print('Đã cài xong. Đang khởi động lại máy ảo — chạy tiếp ô dưới.')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


## 2. Kiểm sau khi khởi động lại


In [ ]:
import fiftyone as fo
import PIL
print('fiftyone', fo.__version__, '| pillow', PIL.__version__)


## 3. Tải gói zip lên

Chạy ô dưới rồi chọn tệp `.zip` từ máy.


In [ ]:
from google.colab import files
import zipfile, pathlib, io

da_tai = files.upload()
ten_zip = next(iter(da_tai))

goc = pathlib.Path('/content/goi_soi')
goc.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(da_tai[ten_zip])) as z:
    z.extractall(goc)

print('đã giải nén:', sorted(p.name for p in goc.iterdir()))


## 4. Nạp vào FiftyOne

Ô này chứa nguyên bản `scripts/nap_fiftyone.py` — chép vào đây để notebook
chạy độc lập, không cần mã nguồn dự án.


In [ ]:
import json
from pathlib import Path
import fiftyone as fo


def nap(thu_muc, ten_bo=None):
    goc = Path(thu_muc)
    with (goc / 'du_lieu.json').open('r', encoding='utf-8') as f:
        goi = json.load(f)

    ten_bo = ten_bo or goi.get('ten') or goc.name
    if ten_bo in fo.list_datasets():
        fo.delete_dataset(ten_bo)

    ds = fo.Dataset(ten_bo, persistent=False)
    mau = []
    for k in goi['khung']:
        s = fo.Sample(filepath=str(goc / 'anh' / k['anh']))
        s['video_id'] = k['video_id']
        s['n'] = k['n']
        s['frame_idx'] = k['frame_idx']          # SỐ PHẢI NỘP
        s['giay'] = round(k['pts_time'], 2)
        s['hang'] = k['hang']
        s['diem'] = round(k['diem'], 5)
        s['nguon'] = k['nguon']

        if k.get('vat_the'):
            s['vat_the'] = fo.Detections(detections=[
                fo.Detection(label=v['nhan'], bounding_box=v['hop'],
                             confidence=v['diem'])
                for v in k['vat_the']
            ])
        if k.get('ocr'):
            s['ocr'] = ' | '.join(b['chu'] for b in k['ocr'] if str(b.get('chu','')).strip())
            s['ocr_conf_cao_nhat'] = max(b['conf'] for b in k['ocr'])
        if 'la_dap_an' in k:
            s['la_dap_an'] = bool(k['la_dap_an'])
        mau.append(s)

    ds.add_samples(mau)
    print(f'{len(mau)} khung')
    if goi.get('cau_hoi'):
        print('Câu hỏi:', goi['cau_hoi'][:120])
    dung = [k['hang'] for k in goi['khung'] if k.get('la_dap_an')]
    if goi.get('khoang_dap_an'):
        if dung:
            print('Khung ĐÚNG ở hạng:', dung)
        elif goi.get('video_dap_an_co_anh') is False:
            print('KHÔNG có khung đúng — video đáp án',
                  goi['khoang_dap_an']['video_id'],
                  'không có ảnh trên máy đã xuất gói.')
            print('Đây là THIẾU SHARD, KHÔNG phải mạch trượt.')
        else:
            print('KHÔNG khung nào trúng khoảng đáp án.')
    if goi.get('so_thieu_anh'):
        print(f"Lưu ý: {goi['so_thieu_anh']}/{goi.get('so_khung_yeu_cau','?')}"
              ' khung bị bỏ vì máy xuất gói không có ảnh.')
    return ds


thu_muc = next(p for p in goc.iterdir() if (p / 'du_lieu.json').exists()) \
          if not (goc / 'du_lieu.json').exists() else goc
ds = nap(thu_muc)


## 5. Mở giao diện

Lưới ảnh hiện ngay dưới ô này. Bấm một ảnh để xem to, hộp vật thể vẽ chồng lên.


In [ ]:
session = fo.launch_app(ds.sort_by('hang'))


## 6. Vài phép lọc hay dùng

Chạy ô tương ứng rồi giao diện tự cập nhật.


In [ ]:
# Chỉ xem khung ĐÚNG (khi gói xuất bằng --dev)
session.view = ds.match({'la_dap_an': True})


In [ ]:
# Khung có vật thể cụ thể
from fiftyone import ViewField as F
session.view = ds.filter_labels('vat_the', F('label') == 'Drum')


In [ ]:
# Khung đọc được chữ, sắp theo độ tin cậy OCR
session.view = ds.exists('ocr').sort_by('ocr_conf_cao_nhat', reverse=True)


In [ ]:
# 50 ứng viên đầu bảng
session.view = ds.sort_by('hang').limit(50)


In [ ]:
# Đếm ứng viên theo video — thấy ngay mạch đang dồn vào video nào
print(ds.count_values('video_id'))


## 7. Ghi lại kết luận

Đánh dấu khung nào là đáp án thật (bấm ảnh → thêm tag `dung`), rồi chạy ô dưới
để lấy ra `video_id` và `frame_idx` — dán thẳng vào bộ dev hoặc tệp nộp.


In [ ]:
da_danh_dau = ds.match_tags('dung')
for s in da_danh_dau:
    print(f"{s.video_id},{s.frame_idx}    # n={s.n} giây={s.giay} hạng={s.hang}")
print(f'\n{len(da_danh_dau)} khung đã đánh dấu')


---
### Nếu máy có sẵn ảnh thì đừng dùng Colab

```powershell
pip install fiftyone
python -u -m scripts.xuat_goi_soi --dev 25 --tai-cho
```
Chạy tại chỗ nhanh hơn, ảnh đầy đủ độ phân giải, và không phải tải gì lên.


---
## Nếu FiftyOne vẫn vỡ — dùng bảng soi HTML

Colab đổi bản dựng sẵn thì xung đột có thể quay lại. Đường này không cài gì cả:

```powershell
python -u -m scripts.dung_bang_soi --goi D:\aic-data\derived\goi_soi\dev_25
```

Sinh ra một tệp `.html` mở bằng trình duyệt: lưới ảnh, hộp vật thể vẽ chồng lên,
chữ OCR, ô lọc, bấm ảnh để chọn rồi chép ra danh sách `video_id,frame_idx`.

Thêm `--nhung-anh` thì ảnh nằm luôn trong tệp HTML — gửi một tệp cho người khác xem được.
